# 05 Visualization and Leaderboard

Create tiny normalized datasets, export a dataset map and pair explanation, then rank algorithm presets with a local leaderboard. The generated dataset ZIPs can also be uploaded to the Gradio Ready Leaderboard tab.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

MATHEEL_REF = "v0.5.3"


def run(*args, cwd=None):
    subprocess.check_call(list(args), cwd=cwd)

run(sys.executable, "-m", "pip", "install", "jedi>=0.16,<1")
run("rm", "-rf", "/content/matheel")
run("git", "clone", "https://github.com/FahadEbrahim/matheel.git", "/content/matheel")
run("git", "checkout", MATHEEL_REF, cwd="/content/matheel")
run(sys.executable, "-m", "pip", "install", "-e", ".[all]", cwd="/content/matheel")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matheel_mplconfig")
print(f"Matheel installed from {MATHEEL_REF}")


In [ ]:
%cd /content/matheel
import json
import shutil
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

from matheel.datasets import write_pair_dataset, write_retrieval_dataset
from matheel.leaderboard import run_leaderboard
from matheel.visualization import build_dataset_embedding_map, write_dataset_map_artifacts, write_pair_dataset_explanation

workspace = Path("/content/matheel_visual_leaderboard_demo")
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir(parents=True)


In [ ]:
pair_root = workspace / "pairs"
write_pair_dataset(
    pair_root,
    files=pd.DataFrame([
        {"file_id": "sum_original", "text": "def total(values):\n    return sum(values)\n", "suffix": ".py", "split": "train"},
        {"file_id": "sum_renamed", "text": "def total(items):\n    return sum(items)\n", "suffix": ".py", "split": "train"},
        {"file_id": "sort_different", "text": "def ordered(values):\n    return sorted(values)\n", "suffix": ".py", "split": "test"},
    ]),
    pairs=pd.DataFrame([
        {"left_id": "sum_original", "right_id": "sum_renamed", "label": 1},
        {"left_id": "sum_original", "right_id": "sort_different", "label": 0},
    ]),
    metadata={"name": "notebook_pairs", "license": "synthetic"},
)

retrieval_root = workspace / "retrieval"
write_retrieval_dataset(
    retrieval_root,
    files=pd.DataFrame([
        {"file_id": "query_sum", "text": "def total(values):\n    return sum(values)\n", "suffix": ".py"},
        {"file_id": "doc_sum", "text": "def total(items):\n    return sum(items)\n", "suffix": ".py"},
        {"file_id": "doc_sort", "text": "def ordered(values):\n    return sorted(values)\n", "suffix": ".py"},
    ]),
    queries=pd.DataFrame([{"query_id": "q_sum", "file_id": "query_sum"}]),
    corpus=pd.DataFrame([
        {"document_id": "d_sum", "file_id": "doc_sum"},
        {"document_id": "d_sort", "file_id": "doc_sort"},
    ]),
    qrels=pd.DataFrame([{"query_id": "q_sum", "document_id": "d_sum", "relevance": 1}]),
    metadata={"name": "notebook_retrieval", "license": "synthetic"},
)

def zip_dir(source, target):
    with zipfile.ZipFile(target, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(source.rglob("*")):
            if path.is_file():
                archive.write(path, arcname=Path(source.name) / path.relative_to(source))
    return target

pair_zip = zip_dir(pair_root, workspace / "pairs.zip")
retrieval_zip = zip_dir(retrieval_root, workspace / "retrieval.zip")
print(pair_zip)
print(retrieval_zip)


In [ ]:
projection = build_dataset_embedding_map(
    pair_root,
    kind="pair",
    method="pca",
    seed=7,
)
map_artifacts = write_dataset_map_artifacts(
    projection,
    workspace / "dataset_map",
    color_column="split",
)
display(projection[["document_id", "split", "x", "y"]].round(4))
display(HTML(map_artifacts["html"].read_text(encoding="utf-8")))


In [ ]:
explanation, explanation_artifacts = write_pair_dataset_explanation(
    pair_root,
    workspace / "pair_explanation",
    left_id="sum_original",
    right_id="sum_renamed",
    segment_mode="line",
)
print(json.dumps(explanation["metadata"], indent=2, sort_keys=True))
display(HTML(explanation_artifacts["html"].read_text(encoding="utf-8")))


In [ ]:
manifest = {
    "name": "notebook_ready_leaderboard",
    "seed": 7,
    "pair_metrics": ["f1", "accuracy", "auroc", "average_precision"],
    "retrieval_metrics": ["mean_average_precision", "mean_reciprocal_rank", "ndcg_at_k"],
    "datasets": [
        {"name": "notebook_pairs", "task": "pair", "path": str(pair_root), "threshold": 0.5},
        {"name": "notebook_retrieval", "task": "retrieval", "path": str(retrieval_root), "k": 2},
    ],
    "algorithms": [
        {"name": "levenshtein", "feature_weights": {"levenshtein": 1.0}},
        {"name": "lexical_blend", "feature_weights": {"levenshtein": 0.5, "winnowing": 0.25, "gst": 0.25}},
    ],
}
report, leaderboard_artifacts = run_leaderboard(
    manifest,
    output_dir=workspace / "leaderboard",
    basename="notebook",
)
display(report["aggregate"].sort_values(["task_family", "metric", "rank", "algorithm_name"]).round(4))
display(report["per_dataset"].sort_values(["task_family", "dataset_name", "metric", "rank", "algorithm_name"]).round(4))
display(HTML(leaderboard_artifacts["html"].read_text(encoding="utf-8")))
print(leaderboard_artifacts["json"])
